# NeuroGolf 2026 EDA

This notebook reviews the ARC-style NeuroGolf task data before modeling. It is designed to run inside Kaggle, where the competition or companion public data is attached under `/kaggle/input`. The analysis focuses on task coverage, pair counts, grid geometry, color-token behavior, visual inspection, and first-pass solver buckets.

# 1. Setup and Data Loading

## 1.1 Imports, Configuration, and Display Defaults

The runtime assumptions are intentionally Kaggle-native: import common notebook libraries, define reusable display constants, and keep plot styling consistent across the analysis.

In [ ]:
from __future__ import annotations

import json
import math
import os
from collections import Counter
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import BoundaryNorm, ListedColormap

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

PLOT_CMAP = "viridis"
plt.style.use("default")
plt.rcParams.update(
    {
        "figure.figsize": (10, 5),
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

ARC_COLORS = [
    "#000000",  # 0 black
    "#0074D9",  # 1 blue
    "#FF4136",  # 2 red
    "#2ECC40",  # 3 green
    "#FFDC00",  # 4 yellow
    "#AAAAAA",  # 5 gray
    "#F012BE",  # 6 magenta
    "#FF851B",  # 7 orange
    "#7FDBFF",  # 8 light blue
    "#870C25",  # 9 maroon
]
ARC_CMAP = ListedColormap(ARC_COLORS)
ARC_NORM = BoundaryNorm(np.arange(-0.5, 10.5, 1), ARC_CMAP.N)

EXPECTED_TASK_COUNT = 400
MAX_DISPLAYED_INPUT_PATHS = 25
MAX_TABLE_ROWS = 20
MAX_SAMPLE_TASKS = 8
MAX_VISUAL_PAIRS = 3
REPORT_FIGURE_DPI = 160


## 1.2 Data Discovery

The input scanner avoids hard-coded Kaggle dataset names. It searches `/kaggle/input` first and then falls back to common local folders for occasional offline development.

In [ ]:
KAGGLE_INPUT = Path("/kaggle/input")
LOCAL_CANDIDATES = [
    Path("../input"),
    Path("input"),
    Path("data"),
    Path("../data"),
]


def candidate_roots() -> list[Path]:
    """Return input roots to scan for NeuroGolf JSON files.

    Returns:
        Candidate directories in priority order."""
    roots: list[Path] = []
    if KAGGLE_INPUT.exists():
        roots.extend([p for p in KAGGLE_INPUT.iterdir() if p.is_dir()])
        roots.append(KAGGLE_INPUT)
    roots.extend([p for p in LOCAL_CANDIDATES if p.exists()])
    return roots


def find_json_files() -> list[Path]:
    """Find task JSON files below candidate input roots.

    Returns:
        Sorted paths to discovered JSON files."""
    files: list[Path] = []
    for root in candidate_roots():
        files.extend(root.rglob("*.json"))
    return sorted(set(files))


json_files = find_json_files()
print(f"Found {len(json_files):,} JSON files")
for path in json_files[:MAX_DISPLAYED_INPUT_PATHS]:
    print(path)
if len(json_files) > MAX_DISPLAYED_INPUT_PATHS:
    print(f"... {len(json_files) - 25:,} more")


### 1.2 Data Discovery Insights

- The expected Kaggle run should discover competition JSON files under `/kaggle/input`.
- If this count is zero, attach the NeuroGolf competition data before continuing.
- Printed paths are a quick sanity check for whether tasks are stored per file or in a combined payload.

## 1.3 Data Loading

NeuroGolf public data can appear as one JSON file per task or as a combined dictionary. This loader normalizes both layouts into a single `tasks` mapping keyed by task id.

In [ ]:
def is_task_payload(obj: Any) -> bool:
    """Check whether a JSON object has the expected task layout.

    Args:
        obj: Parsed JSON object.

    Returns:
        True when the object contains train and test examples."""
    return isinstance(obj, dict) and "train" in obj and "test" in obj


def normalize_task_id(path: Path, key: str | None = None) -> str:
    """Build a stable task id from a JSON path.

    Args:
        path: Source JSON path.

    Returns:
        Normalized task id using the file stem."""
    if key:
        key = str(key)
        return key[:-5] if key.endswith(".json") else key
    return path.stem


def load_tasks(files: list[Path]) -> dict[str, dict[str, Any]]:
    """Load NeuroGolf tasks from one-file or combined JSON layouts.

    Args:
        files: JSON files discovered from the input roots.

    Returns:
        Mapping from task id to normalized task payload."""
    tasks: dict[str, dict[str, Any]] = {}
    for path in files:
        try:
            with path.open("r") as f:
                obj = json.load(f)
        except Exception as exc:
            print(f"Skipping {path}: {exc}")
            continue

        if is_task_payload(obj):
            tasks[normalize_task_id(path)] = obj
        elif isinstance(obj, dict):
            for key, value in obj.items():
                if is_task_payload(value):
                    tasks[normalize_task_id(path, key)] = value
        elif isinstance(obj, list):
            for idx, value in enumerate(obj, start=1):
                if is_task_payload(value):
                    tasks[f"task{idx:03d}"] = value
    return dict(sorted(tasks.items()))


tasks = load_tasks(json_files)
print(f"Loaded {len(tasks):,} tasks")
display(list(tasks)[:10])


### 1.3 Data Loading Insights

- The normalized `tasks` dictionary is the shared foundation for EDA, diagnostics, and model generation.
- The first task ids should follow the `task001` naming convention expected by the ONNX submission.
- A large gap between discovered JSON files and loaded tasks usually means some files are metadata rather than ARC task payloads.

# 2. Dataset Overview

## 2.1 Task Inventory

The inventory converts nested train/test examples into task-level features: pair counts, grid sizes, observed color sets, and whether training outputs change shape.

In [ ]:
def grid_shape(grid: list[list[int]]) -> tuple[int, int]:
    """Return the row and column shape for a grid.

    Args:
        grid: ARC-style integer grid.

    Returns:
        Tuple of row count and column count."""
    arr = np.asarray(grid)
    return tuple(arr.shape) if arr.ndim == 2 else (0, 0)


def grid_colors(grid: list[list[int]]) -> set[int]:
    """Return the unique color tokens used in a grid.

    Args:
        grid: ARC-style integer grid.

    Returns:
        Set of integer color tokens."""
    arr = np.asarray(grid)
    return set(map(int, np.unique(arr))) if arr.size else set()


def task_summary(task_id: str, task: dict[str, Any]) -> dict[str, Any]:
    """Summarize task-level structure for EDA.

    Args:
        task_id: Stable task identifier.
        task: Normalized task payload.

    Returns:
        Dictionary of pair counts, shape features, and color features."""
    train = task.get("train", [])
    test = task.get("test", [])
    pairs = train + test

    input_shapes = [
        grid_shape(pair["input"]) for pair in pairs if "input" in pair
    ]
    output_shapes = [
        grid_shape(pair["output"]) for pair in pairs if "output" in pair
    ]
    train_output_shapes = [
        grid_shape(pair["output"]) for pair in train if "output" in pair
    ]

    input_colors = (
        set().union(
            *(grid_colors(pair["input"]) for pair in pairs if "input" in pair)
        )
        if pairs
        else set()
    )
    output_colors = (
        set().union(
            *(
                grid_colors(pair["output"])
                for pair in pairs
                if "output" in pair
            )
        )
        if output_shapes
        else set()
    )

    train_shape_changes = [
        grid_shape(pair["input"]) != grid_shape(pair["output"])
        for pair in train
        if "input" in pair and "output" in pair
    ]

    return {
        "task_id": task_id,
        "n_train": len(train),
        "n_test": len(test),
        "has_test_outputs": (
            all("output" in pair for pair in test) if test else False
        ),
        "input_shapes": sorted(set(input_shapes)),
        "output_shapes": sorted(set(output_shapes)),
        "train_output_shapes": sorted(set(train_output_shapes)),
        "n_input_colors": len(input_colors),
        "n_output_colors": len(output_colors),
        "input_colors": tuple(sorted(input_colors)),
        "output_colors": tuple(sorted(output_colors)),
        "shape_changes_in_train": any(train_shape_changes),
        "max_input_area": max((r * c for r, c in input_shapes), default=0),
        "max_output_area": max((r * c for r, c in output_shapes), default=0),
    }


summary_df = pd.DataFrame(
    [task_summary(task_id, task) for task_id, task in tasks.items()]
)
display(summary_df.head(MAX_TABLE_ROWS))

if summary_df.empty:
    pass
else:
    pass


### 2.1 Task Inventory Insights

- Use the summary table to identify low-shot tasks, shape-changing tasks, large-grid stress cases, and high-palette tasks.
- `shape_changes_in_train`, `max_input_area`, and color-set columns are the most important solver-planning features.
- This table should remain the source of truth for downstream model manifests.

## 2.2 Coverage Checks

The competition submission format expects task ids such as `task001` through `task400`. This check makes missing or non-standard ids visible before downstream solver notebooks rely on them.

In [ ]:
print(summary_df.shape)
if summary_df.empty:
    print(
        "No tasks loaded yet. Attach the Kaggle competition/public data and rerun from the top."
    )
else:
    display(summary_df.describe(include="all"))

    expected_ids = {f"task{i:03d}" for i in range(1, EXPECTED_TASK_COUNT + 1)}
    observed_ids = set(summary_df["task_id"])
    missing_expected = sorted(expected_ids - observed_ids)
    extra_ids = sorted(observed_ids - expected_ids)

    present_count = len(expected_ids & observed_ids)
    print(
        f"Expected-style tasks present: {present_count:,} / {EXPECTED_TASK_COUNT}"
    )
    missing_preview = missing_expected[:20]
    missing_suffix = " ..." if len(missing_expected) > 20 else ""
    print(f"Missing expected ids: {missing_preview}{missing_suffix}")
    extra_preview = extra_ids[:20]
    extra_suffix = " ..." if len(extra_ids) > 20 else ""
    print(f"Non-standard ids: {extra_preview}{extra_suffix}")


### 2.2 Coverage Insights

- A healthy run should show all expected task ids from `task001` through `task400`.
- Missing ids should be fixed before any ONNX submission work.
- Non-standard ids usually indicate auxiliary JSON files or an unexpected input layout.

# 3. EDA Findings

## 3.1 Pair Distributions

Pair count distributions estimate how much evidence each task provides. Tasks with very few training examples generally need stronger priors and simpler transformation hypotheses.

In [ ]:
if not summary_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    summary_df["n_train"].value_counts().sort_index().plot(
        kind="bar", ax=axes[0], color=plt.get_cmap(PLOT_CMAP)(0.62)
    )
    axes[0].set_title("Training examples per task")
    axes[0].set_xlabel("n_train")
    axes[0].set_ylabel("task count")

    summary_df["n_test"].value_counts().sort_index().plot(
        kind="bar", ax=axes[1], color=plt.get_cmap(PLOT_CMAP)(0.82)
    )
    axes[1].set_title("Test cases per task")
    axes[1].set_xlabel("n_test")
    axes[1].set_ylabel("task count")
    plt.tight_layout()

    train_mode = summary_df["n_train"].mode().iloc[0]
    test_mode = summary_df["n_test"].mode().iloc[0]
else:
    pass


### 3.1 Pair Distribution Insights

- Low train-example counts favor rule induction and hypothesis testing over conventional model training.
- Multiple-test tasks require input-conditioned logic or a selector-style model; a single constant output is not enough.
- Keep this distribution visible when deciding whether a solver can rely on repeated evidence.

## 3.2 Grid Geometry

Grid geometry affects solver design and ONNX cost. Same-shape tasks often suit masking or recoloring logic, while shape-changing tasks need resize, crop, tile, object extraction, or construction strategies.

In [ ]:
if not summary_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    summary_df["max_input_area"].hist(
        ax=axes[0], bins=30, color=plt.get_cmap(PLOT_CMAP)(0.50)
    )
    axes[0].set_title("Max input area")
    axes[0].set_xlabel("cells")
    axes[0].set_ylabel("task count")

    summary_df["max_output_area"].hist(
        ax=axes[1], bins=30, color=plt.get_cmap(PLOT_CMAP)(0.70)
    )
    axes[1].set_title("Max output area")
    axes[1].set_xlabel("cells")

    summary_df["shape_changes_in_train"].value_counts().sort_index().plot(
        kind="bar",
        ax=axes[2],
        color=[plt.get_cmap(PLOT_CMAP)(0.35), plt.get_cmap(PLOT_CMAP)(0.85)],
    )
    axes[2].set_title("Shape changes in training pairs")
    axes[2].set_xlabel("shape changes")
    axes[2].set_ylabel("task count")
    plt.tight_layout()

    largest_tasks = summary_df.sort_values(
        ["max_input_area", "max_output_area"], ascending=False
    ).head(MAX_TABLE_ROWS)
    display(largest_tasks)

else:
    pass


### 3.2 Grid Geometry Insights

- Same-shape and shape-changing tasks should be treated as separate solver tracks.
- Large-grid tasks are useful stress tests for ONNX file size, memory, and graph cost.
- Strong compression and expansion examples should be visually reviewed before implementing shape solvers.

## 3.3 Color Usage

ARC colors are discrete symbols rather than natural-image colors. Frequency and palette-size checks help identify tasks that may be solved by color mapping, background handling, object masks, or simple token substitutions.

In [ ]:
def count_grid_values(
    tasks: dict[str, dict[str, Any]], split: str, field: str
) -> Counter:
    """Count color-token frequency across task grids.

    Args:
        tasks: Mapping of task ids to task payloads.
        split: Task split to scan, such as train or test.
        field: Grid field to count, such as input or output.

    Returns:
        Counter keyed by color token."""
    counts: Counter = Counter()
    for task in tasks.values():
        for pair in task.get(split, []):
            if field in pair:
                counts.update(
                    np.asarray(pair[field]).ravel().astype(int).tolist()
                )
    return counts


train_input_color_counts = count_grid_values(tasks, "train", "input")
train_output_color_counts = count_grid_values(tasks, "train", "output")

color_df = pd.DataFrame(
    {
        "color": range(10),
        "train_input_cells": [
            train_input_color_counts.get(i, 0) for i in range(10)
        ],
        "train_output_cells": [
            train_output_color_counts.get(i, 0) for i in range(10)
        ],
    }
)
display(color_df)

if not color_df.empty:
    ax = color_df.set_index("color")[
        ["train_input_cells", "train_output_cells"]
    ].plot(kind="bar", figsize=(10, 4), colormap=PLOT_CMAP)
    ax.set_title("ARC token frequency in training pairs")
    ax.set_xlabel("ARC color token")
    ax.set_ylabel("cell count")
    plt.tight_layout()

if not summary_df.empty:
    ax = (
        summary_df["n_input_colors"]
        .value_counts()
        .sort_index()
        .plot(kind="bar", figsize=(8, 4), color=plt.get_cmap(PLOT_CMAP)(0.68))
    )
    ax.set_title("Unique input colors per task")
    ax.set_xlabel("unique colors")
    ax.set_ylabel("task count")
    plt.tight_layout()

    dominant_input = int(
        color_df.sort_values("train_input_cells", ascending=False).iloc[0][
            "color"
        ]
    )
    dominant_output = int(
        color_df.sort_values("train_output_cells", ascending=False).iloc[0][
            "color"
        ]
    )
else:
    pass


### 3.3 Color Usage Insights

- Color `0` is dominant in the current benchmark and should be treated as a likely background candidate, not an assumption.
- Palette size helps separate simple recoloring from richer object or pattern tasks.
- Compare input and output frequencies to find colors that are introduced, removed, or preserved.

## 3.4 Structural Deep Dive

This diagnostic separates tasks by output-shape behavior, area expansion/compression, and multiple-test-case risk. These features are useful for deciding whether a baseline can be task-constant, same-shape, or needs input-conditioned logic.

In [ ]:
if not summary_df.empty:
    deep_df = summary_df.copy()
    deep_df["area_delta"] = (
        deep_df["max_output_area"] - deep_df["max_input_area"]
    )
    deep_df["area_ratio"] = np.where(
        deep_df["max_input_area"] > 0,
        deep_df["max_output_area"] / deep_df["max_input_area"],
        np.nan,
    )
    deep_df["test_case_group"] = np.where(
        deep_df["n_test"] == 1, "single test case", "multiple test cases"
    )
    deep_df["area_group"] = pd.cut(
        deep_df["area_ratio"],
        bins=[-np.inf, 0.5, 0.95, 1.05, 2.0, np.inf],
        labels=[
            "strong compression",
            "mild compression",
            "same area",
            "mild expansion",
            "strong expansion",
        ],
    )

    structural_table = pd.crosstab(
        deep_df["area_group"], deep_df["shape_changes_in_train"], margins=True
    )
    test_table = (
        deep_df["test_case_group"]
        .value_counts()
        .rename_axis("test_case_group")
        .reset_index(name="task_count")
    )
    display(structural_table)
    display(test_table)
    display(
        deep_df.sort_values("area_ratio", ascending=False).head(
            min(10, MAX_TABLE_ROWS)
        )[
            [
                "task_id",
                "n_train",
                "n_test",
                "input_shapes",
                "output_shapes",
                "area_ratio",
                "area_group",
            ]
        ]
    )
    display(
        deep_df.sort_values("area_ratio", ascending=True).head(
            min(10, MAX_TABLE_ROWS)
        )[
            [
                "task_id",
                "n_train",
                "n_test",
                "input_shapes",
                "output_shapes",
                "area_ratio",
                "area_group",
            ]
        ]
    )

    single_test_share = (deep_df["n_test"] == 1).mean()
    same_area_share = (deep_df["area_group"] == "same area").mean()
else:
    pass


### 3.4 Structural Deep Dive Insights

- Area groups identify solver families: same-area, compression, expansion, and extreme shape changes.
- Strong compression often points to crop, extract, select, summarize, or count behavior.
- Strong expansion often points to scale, tile, draw, complete, or construct behavior.

## 3.5 Color Delta Deep Dive

Color set differences between inputs and outputs reveal tasks that introduce new tokens, remove tokens, or preserve the same palette while changing geometry or layout.

In [ ]:
if not summary_df.empty:
    color_delta_df = summary_df.copy()
    color_delta_df["introduced_colors"] = color_delta_df.apply(
        lambda row: tuple(
            sorted(set(row["output_colors"]) - set(row["input_colors"]))
        ),
        axis=1,
    )
    color_delta_df["removed_colors"] = color_delta_df.apply(
        lambda row: tuple(
            sorted(set(row["input_colors"]) - set(row["output_colors"]))
        ),
        axis=1,
    )
    color_delta_df["palette_relation"] = np.select(
        [
            color_delta_df["introduced_colors"].map(len).eq(0)
            & color_delta_df["removed_colors"].map(len).eq(0),
            color_delta_df["introduced_colors"].map(len).gt(0)
            & color_delta_df["removed_colors"].map(len).eq(0),
            color_delta_df["introduced_colors"].map(len).eq(0)
            & color_delta_df["removed_colors"].map(len).gt(0),
        ],
        ["same palette", "introduces colors", "removes colors"],
        default="introduces and removes colors",
    )

    palette_table = (
        color_delta_df["palette_relation"]
        .value_counts()
        .rename_axis("palette_relation")
        .reset_index(name="task_count")
    )
    display(palette_table)
    display(
        color_delta_df.query("palette_relation != 'same palette'").head(
            MAX_TABLE_ROWS
        )[
            [
                "task_id",
                "input_colors",
                "output_colors",
                "introduced_colors",
                "removed_colors",
                "shape_changes_in_train",
            ]
        ]
    )

    ax = palette_table.set_index("palette_relation")["task_count"].plot(
        kind="bar", figsize=(8, 4), color=plt.get_cmap(PLOT_CMAP)(0.72)
    )
    ax.set_title("Input-output palette relation")
    ax.set_xlabel("palette relation")
    ax.set_ylabel("task count")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()

    same_palette_share = (
        color_delta_df["palette_relation"] == "same palette"
    ).mean()
else:
    pass


### 3.5 Color Delta Insights

- Same-palette tasks are often geometry, movement, or selection problems.
- Introduced-color tasks suggest marking, filling, completion, or derived labels.
- Removed-color tasks suggest masking, filtering, object selection, or background normalization.

# 4. Task Review

## 4.1 Visual Inspection Tools

Manual inspection remains essential for ARC-style transformations. The helper below renders train and test examples consistently, while gracefully handling missing test outputs.

In [ ]:
def show_grid(ax: plt.Axes, grid: list[list[int]], title: str) -> None:
    """Render one ARC grid on a matplotlib axis.

    Args:
        ax: Axis to draw into.
        grid: ARC-style integer grid.
        title: Short panel title."""
    arr = np.asarray(grid)
    ax.imshow(arr, cmap=ARC_CMAP, norm=ARC_NORM)
    ax.set_title(title, fontsize=10)
    ax.set_xticks(np.arange(-0.5, arr.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, arr.shape[0], 1), minor=True)
    ax.grid(which="minor", color="#555555", linewidth=0.5, alpha=0.55)
    ax.tick_params(
        which="both",
        left=False,
        bottom=False,
        labelleft=False,
        labelbottom=False,
    )


def show_task(task_id: str, max_pairs: int | None = None) -> None:
    """Render train and test examples for one task.

    Args:
        task_id: Task id to render.
        max_pairs: Maximum train/test pairs to show."""
    task = tasks[task_id]
    rows: list[tuple[str, int, dict[str, Any]]] = []
    for split in ["train", "test"]:
        pairs = task.get(split, [])
        if max_pairs is not None:
            pairs = pairs[:max_pairs]
        rows.extend(
            (split, idx, pair) for idx, pair in enumerate(pairs, start=1)
        )

    n_rows = len(rows)
    n_cols = 2
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(6, max(2.2, 2.4 * n_rows))
    )
    axes = np.asarray(axes).reshape(n_rows, n_cols)
    fig.suptitle(task_id, fontsize=14, y=1.0)

    for row_idx, (split, idx, pair) in enumerate(rows):
        show_grid(axes[row_idx, 0], pair["input"], f"{split} {idx} input")
        if "output" in pair:
            show_grid(
                axes[row_idx, 1], pair["output"], f"{split} {idx} output"
            )
        else:
            axes[row_idx, 1].axis("off")
            axes[row_idx, 1].set_title(
                f"{split} {idx} output unavailable", fontsize=10
            )
    plt.tight_layout()


if tasks:
    first_task_id = next(iter(tasks))
    show_task(first_task_id)
else:
    pass


### 4.1 Visual Inspection Insights

- The first rendered task is a smoke test for the visualization helper.
- Use visual review to identify object movement, color replacement, cropping, symmetry, tiling, and counting transformations.
- Visual patterns should become explicit solver hypotheses rather than informal observations.

## 4.2 Diverse Task Samples

The sample set intentionally mixes large grids, many-color tasks, shape-changing tasks, and compact same-shape tasks. This makes browsing more representative than simply looking at the first few task ids.

In [ ]:
sample_ids: list[str] = []
if not summary_df.empty:
    sample_ids.extend(
        summary_df.sort_values("max_input_area", ascending=False)
        .head(2)["task_id"]
        .tolist()
    )
    sample_ids.extend(
        summary_df.sort_values("n_input_colors", ascending=False)
        .head(2)["task_id"]
        .tolist()
    )
    sample_ids.extend(
        summary_df.query("shape_changes_in_train == True")
        .head(2)["task_id"]
        .tolist()
    )
    sample_ids.extend(
        summary_df.query("shape_changes_in_train == False")
        .sort_values("max_input_area")
        .head(2)["task_id"]
        .tolist()
    )

sample_ids = list(dict.fromkeys(sample_ids))
display(sample_ids)


### 4.2 Sampling Insights

- The sampled task list intentionally mixes large grids, many-color tasks, shape-changing tasks, and compact same-shape tasks.
- Review these before implementing solvers so the first solver families are not biased toward early task ids.

In [ ]:
for task_id in sample_ids[:MAX_SAMPLE_TASKS]:
    show_task(task_id, max_pairs=MAX_VISUAL_PAIRS)


### 4.2 Manual Review Notes

- Look for repeated transformation motifs across sampled tasks.
- Flag tasks that cannot be explained by same-shape recoloring, object extraction, or simple geometric construction.
- Convert recurring motifs into solver-family candidates.

## 4.3 Difficult Task Gallery

The diverse sample set is useful for quick visual checks, but solver planning also needs deliberately difficult examples. This gallery renders representative stress cases across expansion, compression, large-grid, rich-palette, multi-test, and palette-change behavior.

In [ ]:
gallery_specs: list[tuple[str, str]] = []

if "deep_df" in globals() and not deep_df.empty:
    expansion_df = deep_df.query(
        "area_group == 'strong expansion'"
    ).sort_values("area_ratio", ascending=False)
    compression_df = deep_df.query(
        "area_group == 'strong compression'"
    ).sort_values("area_ratio", ascending=True)
    if not expansion_df.empty:
        gallery_specs.append(
            (
                str(expansion_df.iloc[0]["task_id"]),
                "strong expansion stress case",
            )
        )
    if not compression_df.empty:
        gallery_specs.append(
            (
                str(compression_df.iloc[0]["task_id"]),
                "strong compression stress case",
            )
        )

if not summary_df.empty:
    largest_task_id = summary_df.sort_values(
        ["max_input_area", "max_output_area"], ascending=False
    ).iloc[0]["task_id"]
    gallery_specs.append((str(largest_task_id), "largest-grid stress case"))

    rich_palette_task_id = summary_df.sort_values(
        ["n_input_colors", "n_output_colors"], ascending=False
    ).iloc[0]["task_id"]
    gallery_specs.append(
        (str(rich_palette_task_id), "rich-palette stress case")
    )

    multi_test_df = summary_df.query("n_test > 1").sort_values(
        ["n_test", "max_input_area"], ascending=False
    )
    if not multi_test_df.empty:
        gallery_specs.append(
            (str(multi_test_df.iloc[0]["task_id"]), "multi-test stress case")
        )

if "color_delta_df" in globals() and not color_delta_df.empty:
    mixed_palette_df = color_delta_df.query(
        "palette_relation == 'introduces and removes colors'"
    )
    if not mixed_palette_df.empty:
        gallery_specs.append(
            (
                str(mixed_palette_df.iloc[0]["task_id"]),
                "mixed-palette stress case",
            )
        )

seen_gallery_tasks: set[str] = set()
gallery_rows: list[dict[str, str]] = []
for task_id, reason in gallery_specs:
    if task_id in seen_gallery_tasks or task_id not in tasks:
        continue
    seen_gallery_tasks.add(task_id)
    gallery_rows.append({"task_id": task_id, "reason": reason})

stress_gallery_df = pd.DataFrame(gallery_rows)
display(stress_gallery_df)

for row in stress_gallery_df.head(MAX_SAMPLE_TASKS).itertuples(index=False):
    print(f"{row.task_id}: {row.reason}")
    show_task(row.task_id, max_pairs=MAX_VISUAL_PAIRS)


### 4.3 Difficult Gallery Insights

- Expansion and compression stress cases help separate output-shape construction from object extraction.
- Large-grid and rich-palette examples are useful for checking whether a proposed solver will stay compact and robust in ONNX.
- Multi-test examples should be reviewed before submission modeling because they expose whether a model conditions on the input or only memorizes one output.

# 5. Planning and Artifacts

## 5.1 Baseline Feasibility

Before building solver families, estimate what a submission-format sanity baseline can cover. This is not meant to be the final modeling approach; it checks how much of the dataset can be packaged with simple ONNX output behavior.

In [ ]:
if not summary_df.empty:
    baseline_df = summary_df.copy()
    baseline_df["constant_output_feasible"] = (
        baseline_df["n_test"].eq(1) & baseline_df["has_test_outputs"]
    )
    feasibility_table = (
        baseline_df[["constant_output_feasible", "shape_changes_in_train"]]
        .value_counts()
        .rename_axis(["constant_output_feasible", "shape_changes_in_train"])
        .reset_index(name="task_count")
    )
    display(feasibility_table)

    feasible_count = int(baseline_df["constant_output_feasible"].sum())
else:
    pass


### 5.1 Baseline Feasibility Insights

- Constant-output ONNX models are useful for packaging validation, not for a final general solution.
- Multiple-test tasks are the first pressure test for input-conditioned behavior.
- Baseline feasibility should be interpreted as submission-mechanics coverage, not modeling quality.

## 5.2 Solver Planning Buckets

These buckets are deliberately simple. They create a practical bridge from EDA to later solver notebooks, where each bucket can be replaced by measured solver diagnostics and ONNX export checks.

In [ ]:
def assign_bucket(row: pd.Series) -> str:
    """Assign a provisional solver-planning bucket.

    Args:
        row: Task summary row.

    Returns:
        Solver bucket label for planning."""
    if row["shape_changes_in_train"]:
        return "shape-changing"
    if row["n_input_colors"] <= 2:
        return "low-color same-shape"
    if row["max_input_area"] <= 100:
        return "small same-shape"
    return "larger same-shape"


if not summary_df.empty:
    summary_df = summary_df.copy()
    summary_df["eda_bucket"] = summary_df.apply(assign_bucket, axis=1)
    bucket_df = (
        summary_df["eda_bucket"]
        .value_counts()
        .rename_axis("bucket")
        .reset_index(name="task_count")
    )
    display(bucket_df)

    ax = (
        summary_df["eda_bucket"]
        .value_counts()
        .plot(kind="bar", figsize=(9, 4), color=plt.get_cmap(PLOT_CMAP)(0.74))
    )
    ax.set_title("EDA task buckets for solver planning")
    ax.set_xlabel("bucket")
    ax.set_ylabel("task count")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()

    display(summary_df.head(min(5, MAX_TABLE_ROWS)))
    leading_bucket = bucket_df.iloc[0]
else:
    pass


### 5.2 Solver Planning Insights

- The provisional buckets define the first modeling tracks.
- Track pass rates by bucket once real solvers are implemented.
- Prioritize solver families that explain all train pairs and can be exported compactly to ONNX.

## 5.3 Report Figures and Markdown Assets

This section saves publication-ready EDA figures and a markdown report fragment. These assets make it easier to carry the latest notebook results into project notes, README updates, or Kaggle discussion writeups without manually recreating plots.

In [ ]:
REPORT_OUTPUT_DIR = (
    Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
)
REPORT_DIR = REPORT_OUTPUT_DIR / "eda_report_figures"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

saved_figures: list[dict[str, str]] = []


def save_current_figure(filename: str, title: str, caption: str) -> None:
    """Save the active figure and register its report metadata.

    Args:
        filename: Figure filename under the report directory.
        title: Human-readable report title.
        caption: Short analytical caption."""
    path = REPORT_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=REPORT_FIGURE_DPI, bbox_inches="tight")
    plt.close()
    saved_figures.append(
        {"title": title, "path": str(path), "caption": caption}
    )


if not summary_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    summary_df["n_train"].value_counts().sort_index().plot(
        kind="bar", ax=axes[0], color=plt.get_cmap(PLOT_CMAP)(0.62)
    )
    axes[0].set_title("Training examples per task")
    axes[0].set_xlabel("n_train")
    axes[0].set_ylabel("task count")
    summary_df["n_test"].value_counts().sort_index().plot(
        kind="bar", ax=axes[1], color=plt.get_cmap(PLOT_CMAP)(0.82)
    )
    axes[1].set_title("Test cases per task")
    axes[1].set_xlabel("n_test")
    axes[1].set_ylabel("task count")
    save_current_figure(
        "1_pair_distributions.png",
        "Pair Distributions",
        "Most tasks are low-shot and single-test, which supports "
        "rule search over per-task neural training.",
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    summary_df["max_input_area"].hist(
        ax=axes[0], bins=30, color=plt.get_cmap(PLOT_CMAP)(0.50)
    )
    axes[0].set_title("Max input area")
    axes[0].set_xlabel("cells")
    axes[0].set_ylabel("task count")
    summary_df["max_output_area"].hist(
        ax=axes[1], bins=30, color=plt.get_cmap(PLOT_CMAP)(0.70)
    )
    axes[1].set_title("Max output area")
    axes[1].set_xlabel("cells")
    summary_df["shape_changes_in_train"].value_counts().sort_index().plot(
        kind="bar",
        ax=axes[2],
        color=[plt.get_cmap(PLOT_CMAP)(0.35), plt.get_cmap(PLOT_CMAP)(0.85)],
    )
    axes[2].set_title("Shape changes in training pairs")
    axes[2].set_xlabel("shape changes")
    axes[2].set_ylabel("task count")
    save_current_figure(
        "2_grid_geometry.png",
        "Grid Geometry",
        "Shape-changing tasks are large enough to deserve a dedicated solver track.",
    )

if "color_df" in globals() and not color_df.empty:
    ax = color_df.set_index("color")[
        ["train_input_cells", "train_output_cells"]
    ].plot(kind="bar", figsize=(10, 4), colormap=PLOT_CMAP)
    ax.set_title("ARC token frequency in training pairs")
    ax.set_xlabel("ARC color token")
    ax.set_ylabel("cell count")
    save_current_figure(
        "3_color_frequency.png",
        "Color Frequency",
        (
            "Color 0 dominates the benchmark, making background detection "
            "and manipulation central primitives."
        ),
    )

if "deep_df" in globals() and not deep_df.empty:
    area_counts = deep_df["area_group"].value_counts().sort_index()
    ax = area_counts.plot(
        kind="bar", figsize=(9, 4), color=plt.get_cmap(PLOT_CMAP)(0.68)
    )
    ax.set_title("Area transformation groups")
    ax.set_xlabel("area group")
    ax.set_ylabel("task count")
    plt.xticks(rotation=25, ha="right")
    save_current_figure(
        "4_area_groups.png",
        "Area Transformation Groups",
        (
            "Compression and expansion groups highlight where crop, extract, "
            "tile, and construct solvers are needed."
        ),
    )

if "color_delta_df" in globals() and not color_delta_df.empty:
    palette_table = color_delta_df["palette_relation"].value_counts()
    ax = palette_table.plot(
        kind="bar", figsize=(8, 4), color=plt.get_cmap(PLOT_CMAP)(0.72)
    )
    ax.set_title("Input-output palette relation")
    ax.set_xlabel("palette relation")
    ax.set_ylabel("task count")
    plt.xticks(rotation=25, ha="right")
    save_current_figure(
        "5_palette_relation.png",
        "Palette Relation",
        (
            "Palette deltas separate geometry-heavy tasks from tasks that "
            "introduce, remove, or relabel colors."
        ),
    )

if "sample_ids" in globals() and sample_ids:
    sample_task_id = sample_ids[0]
    show_task(sample_task_id, max_pairs=MAX_VISUAL_PAIRS)
    save_current_figure(
        "6_sample_task.png",
        f"Sample Task: {sample_task_id}",
        "A representative rendered task sample for visual review of train and test examples.",
    )

difficult_specs: list[tuple[str, str, str]] = []
if "deep_df" in globals() and not deep_df.empty:
    expansion_df = deep_df.query(
        "area_group == 'strong expansion'"
    ).sort_values("area_ratio", ascending=False)
    compression_df = deep_df.query(
        "area_group == 'strong compression'"
    ).sort_values("area_ratio", ascending=True)
    if not expansion_df.empty:
        task_id = expansion_df.iloc[0]["task_id"]
        difficult_specs.append(
            (
                str(task_id),
                "7_difficult_strong_expansion.png",
                "Difficult Sample: Strong Expansion",
            )
        )
    if not compression_df.empty:
        task_id = compression_df.iloc[0]["task_id"]
        difficult_specs.append(
            (
                str(task_id),
                "8_difficult_strong_compression.png",
                "Difficult Sample: Strong Compression",
            )
        )

if not summary_df.empty:
    largest_task = summary_df.sort_values(
        ["max_input_area", "max_output_area"], ascending=False
    ).iloc[0]["task_id"]
    difficult_specs.append(
        (
            str(largest_task),
            "9_difficult_largest_grid.png",
            "Difficult Sample: Largest Grid",
        )
    )

    rich_color_task = summary_df.sort_values(
        ["n_input_colors", "n_output_colors"], ascending=False
    ).iloc[0]["task_id"]
    difficult_specs.append(
        (
            str(rich_color_task),
            "10_difficult_rich_palette.png",
            "Difficult Sample: Rich Palette",
        )
    )

    multi_test_df = summary_df.query("n_test > 1").sort_values(
        ["n_test", "max_input_area"], ascending=False
    )
    if not multi_test_df.empty:
        multi_test_task = multi_test_df.iloc[0]["task_id"]
        difficult_specs.append(
            (
                str(multi_test_task),
                "11_difficult_multi_test.png",
                "Difficult Sample: Multi-Test Task",
            )
        )

seen_difficult: set[str] = set()
for task_id, filename, title in difficult_specs:
    if task_id in seen_difficult or task_id not in tasks:
        continue
    seen_difficult.add(task_id)
    show_task(task_id, max_pairs=MAX_VISUAL_PAIRS)
    save_current_figure(
        filename,
        f"{title}: {task_id}",
        (
            "A deliberately selected hard task sample used to guide solver "
            "design and failure analysis."
        ),
    )

report_lines = [
    "# NeuroGolf 2026 EDA Figures",
    "",
    (
        "This report fragment is generated by `notebooks/1_eda.ipynb` "
        "and points to the latest saved EDA figures."
    ),
    "",
]
for item in saved_figures:
    relative_path = Path(item["path"]).name
    report_lines.extend(
        [
            f"## {item['title']}",
            "",
            item["caption"],
            "",
            f"![{item['title']}]({relative_path})",
            "",
        ]
    )

report_path = REPORT_DIR / "eda-insights-with-figures.md"
report_path.write_text("\n".join(report_lines))

figure_manifest_df = pd.DataFrame(saved_figures)
figure_manifest_path = REPORT_DIR / "eda_figure_manifest.csv"
figure_manifest_df.to_csv(figure_manifest_path, index=False)

display(figure_manifest_df)
print(f"Wrote {report_path}")
print(f"Wrote {figure_manifest_path}")


### 5.3 Report Asset Insights

- The saved figures make EDA results reusable outside the notebook.
- Difficult-task samples anchor solver discussions around concrete examples.
- Refresh these assets after major EDA or solver-diagnostic changes.

## 5.4 Export EDA Artifacts

The final cell writes compact CSV summaries under `/kaggle/working` on Kaggle. These artifacts keep later notebooks lightweight and provide a stable reference for solver planning.

In [ ]:
OUTPUT_DIR = (
    Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
)
if not summary_df.empty:
    summary_path = OUTPUT_DIR / "neurogolf_eda_task_summary.csv"
    color_path = OUTPUT_DIR / "neurogolf_eda_color_counts.csv"
    summary_df.to_csv(summary_path, index=False)
    color_df.to_csv(color_path, index=False)
    print(f"Wrote {summary_path}")
    print(f"Wrote {color_path}")
else:
    pass


### 5.4 Export Insights

- The exported CSVs are lightweight handoff artifacts for solver-development notebooks.
- Keep these files stable so downstream notebooks do not need to recompute every EDA feature.
- If no files are written, rerun from the data-loading section and check input attachment.